# Data Mexico

Ya tenemos la información de la producción agricola a nivel nacional y municipal. Lo que haremos ahora es extraer información de la exportación e importación de los mismos productos. Esto para ver si la producción nacional es lo suficiente para 

* Tener buena ganacia mediante exportación.
* Minimizar la importación. (Idealmente no deberiamos importar nuestros mejores productos agricolas)

Para ello ocupamos cierta información de la Secretaria de Economia.
* Balanza comercial. (Revisar el deficit de cada producto)
* Cantidad importada en Toneladas
* Cantidad exportada en Toneladas
* Precio de Venta y Precio de Compra. (A que precio vendemos y cuanto nos cuesta importar los productos agricolas)

Esperamos obtener toda esta información mediante DataMexico. Una herramienta desarrollada por la propia Secretaria de Economia para el acceso de sus bases de datos. Aunque posiblemente usaremos información del INEGI para información faltante y/o calidad de Datos.

Esto porque en México la información de "mejor calidad" es la otorgada por el INEGI.

## Diccionario de Datos

Al igual que en el DGSIAP. Lo primero es obtener el diccionario de datos respectivo para poder utilizar sus conjuntos de datos de forma adecuada.

Como DataMexico no tiene porque usar la misma estructura que el DGSIAP debemos analizar su API para poder encontrarlo. En este caso, gracias a la organización de DataMexico, todos los diccionarios estan en un solo archivo que es facilmente obtenible.

In [1]:
import yarl
import requests
from scripts import download as d
datadict_url = "https://www.economia.gob.mx/datamexico/files/diccionario.xlsx"
tigie_to_scian = "https://www.inegi.org.mx/contenidos/app/tigie/tabla-correlacion-actualizacion-2025.xlsx?v=0.0.1" 
DATAMEXICO_DICT = yarl.URL(datadict_url)
TIGIE_CATALOG = yarl.URL(tigie_to_scian)

filename_1 = datadict_url.split("/")[-1]
filename_2 = tigie_to_scian.split("/")[-1].split("?")[0]


d.download(DATAMEXICO_DICT, filename_1)
d.download(TIGIE_CATALOG, filename_2)


El archivo ya fue descargado. Se encuentra en /data/external/diccionario.xlsx
El archivo ya fue descargado. Se encuentra en /data/external/tabla-correlacion-actualizacion-2025.xlsx


### TIGIE
El TIGIE o Tarifa de la Ley de los Impuestos Generales de Importación y de Exportación es el documento oficial en México que establece los aranceles y las normas para clasificar todas las mercancías que entran o salen del país. 

#### Catalogo
Ellos poseen un catalogo, que descargamos como  "tabla-correlacion-actualizacion-2025.xlsx" donde etiquetan cada producto que Mexico somete a exportación e importación.

In [ ]:
import pandas as pd
from scripts.download import DATA_PATH

tigie_scian = pd.read_excel(DATA_PATH / filename_2, skiprows=2)
print(tigie_scian.head(n=3))
# Vemos las columnas para saber donde hacer el corte
print(tigie_scian.columns) 

tigie_scian = tigie_scian.drop(axis=0, columns=['Unnamed: 2'])
tigie_scian.head(n=5)

      TIGIE 2022                        Unnamed: 1  Unnamed: 2  \
0         Código                            Nombre         NaN   
1  0101.21.01.00       Reproductores de raza pura.         NaN   
2  0101.29.02.00  Sin pedigree, para reproducción.         NaN   

  SCIAN México 2023               Unnamed: 4  
0            Código                   Nombre  
1            112920  Explotación de équidos   
2            112920  Explotación de équidos   
Index(['TIGIE 2022', 'Unnamed: 1', 'Unnamed: 2', 'SCIAN México 2023',
       'Unnamed: 4'],
      dtype='str')


,TIGIE 2022,Unnamed: 1,SCIAN México 2023,Unnamed: 4
0,Código,Nombre,Código,Nombre
1,0101.21.01.00,Reproductores de raza pura.,112920,Explotación de équidos
2,0101.29.02.00,"Sin pedigree, para reproducción.",112920,Explotación de équidos
3,0101.29.03.00,"Para abasto, cuando la importación la realicen...",112920,Explotación de équidos
4,0101.29.99.01,Para saltos o carreras.,112920,Explotación de équidos


Como podemos ver. El documento tambien incluye su equivalencia con el catalogo del SCIAN (Sistema de Clasificación Industrial de América del Norte), que es relativamente lo mismo pero sobre las actividades economicas. Entonces de este documento podemos obtener conexiones entre las importaciones y exportaciones del pais y la inversión extranjera.

In [3]:
base_cols = set(tigie_scian.loc[0])
print(base_cols)
suffix = list(tigie_scian.columns)[0::2]
print(suffix)
new_cols = [f"{base}_{inst.split(" ")[0]}-{inst.split(" ")[-1]}" for inst in suffix for base in base_cols]
new_cols[-2] = f"{new_cols[-2]}_MX"
new_cols[-1] = f"{new_cols[-1]}_MX"
print(new_cols)

# Colocamos las nuevas columnas

tigie_scian.columns = new_cols
tigie_scian = tigie_scian.iloc[1:]
tigie_scian

{'Código', 'Nombre'}
['TIGIE 2022', 'SCIAN México 2023']
['Código_TIGIE-2022', 'Nombre_TIGIE-2022', 'Código_SCIAN-2023_MX', 'Nombre_SCIAN-2023_MX']


,Código_TIGIE-2022,Nombre_TIGIE-2022,Código_SCIAN-2023_MX,Nombre_SCIAN-2023_MX
1,0101.21.01.00,Reproductores de raza pura.,112920,Explotación de équidos
2,0101.29.02.00,"Sin pedigree, para reproducción.",112920,Explotación de équidos
3,0101.29.03.00,"Para abasto, cuando la importación la realicen...",112920,Explotación de équidos
4,0101.29.99.01,Para saltos o carreras.,112920,Explotación de équidos
5,0101.29.99.99,Los demás.,112920,Explotación de équidos
...,...,...,...,...
11491,9705.31.99.00,Las demás.,339999,Otras industrias manufactureras
11492,9705.39.01.00,"Objetos de interés histórico, que no hayan sid...",339999,Otras industrias manufactureras
11493,9705.39.99.00,Las demás.,339999,Otras industrias manufactureras
11494,9706.10.01.00,De más de 250 años.,339999,Otras industrias manufactureras


## Tesseract Cubes

La api de DATA MEXICO usa Data Cubes para almacenar sus datos. Lo que implica una nueva estructura de datos. Distinta al del DGSIAP que se trabajo mediante HTML Scrapping.

Usando su herramienta visual VizBuilder pudimos obtener un ejemplo. Esta URL nos esta dando los primeros 50 valores de la inversión extranjera de México por Pais/Continente de origen de forma anual.

In [10]:
API_URL_EX = "https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_year_industry_country&drilldowns=Country%2CContinent%2CYear%2CIndustry+Group&limit=50&measures=Investment%2CCount"

q = requests.get(yarl.URL(API_URL_EX))
print(q)

m = q.content
print(m)

KeyboardInterrupt: 

In [ ]:
# Diccionario de Datos
fdi = "https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_year_industry_country&drilldowns=Year%2CCountry%2CIndustry+Group&measures=Investment%2CCount"
fdi = "https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords?cube=fdi_year_industry_country&drilldowns=Year%2CCountry%2CIndustry+Group&locale=es&measures=Investment%2CCount"

tigie_scian = "https://www.inegi.org.mx/contenidos/app/tigie/tabla-correlacion-actualizacion-2025.xlsx?v=0.0.1"
